# Titanic — Feature Engineering
**Goal:** Transform raw data into meaningful features that a Machine Learning model can understand.

## 1. Load Cleaned Data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('Titanic-Dataset.csv') # Loading raw to get Name column back
print('Shape:', df.shape)

Shape: (891, 12)


## 2. Extract Title from Name

In [2]:
df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\\.', expand=False)
df['Title'] = df['Title'].replace(['Lady', 'Countess','Capt', 'Col','Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Other')
df['Title'] = df['Title'].replace('Mlle', 'Miss').replace('Ms', 'Miss').replace('Mme', 'Mrs')
print(df['Title'].value_counts())

Title
Mr        517
Miss      185
Mrs       126
Master     40
Other      23
Name: count, dtype: int64


## 3. Family Features

In [3]:
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
df[['FamilySize', 'IsAlone']].head()

,FamilySize,IsAlone
0,2,0
1,2,0
2,1,1
3,2,0
4,1,1


## 4. Age Binning & Fare Transformation

In [4]:
df['Age'] = df.groupby(['Pclass', 'Title'])['Age'].transform(lambda x: x.fillna(x.median()))
df['AgeBin'] = pd.cut(df['Age'], bins=[0, 12, 18, 35, 60, 100], labels=['Child', 'Teen', 'Adult', 'Middle-Aged', 'Senior'])
df['LogFare'] = np.log1p(df['Fare'])
df[['AgeBin', 'LogFare']].head()

,AgeBin,LogFare
0,Adult,2.110213
1,Middle-Aged,4.280593
2,Adult,2.188856
3,Adult,3.990834
4,Adult,2.202765


## 5. Encoding Categorical Data

In [5]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
df['AgeBin'] = le.fit_transform(df['AgeBin'])

# One-Hot encode Embarked and Title
df = pd.get_dummies(df, columns=['Embarked', 'Title'], drop_first=True)

# Convert bool to int
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

## 6. Final Cleanup & Save

In [6]:
cols_to_drop = ['PassengerId', 'Name', 'Ticket', 'Cabin', 'SibSp', 'Parch', 'Fare']
df.drop(columns=cols_to_drop, inplace=True)
df.to_csv('Titanic-Featured.csv', index=False)
print('Saved Featured Data. Shape:', df.shape)
df.head()

Saved Featured Data. Shape: (891, 14)


,Survived,Pclass,Sex,Age,FamilySize,IsAlone,AgeBin,LogFare,Embarked_Q,Embarked_S,Title_Miss,Title_Mr,Title_Mrs,Title_Other
0,0,3,0,22.0,2,0,0,2.110213,0,1,0,1,0,0
1,1,1,1,38.0,2,0,2,4.280593,0,0,0,0,1,0
2,1,3,1,26.0,1,1,0,2.188856,0,1,1,0,0,0
3,1,1,1,35.0,2,0,0,3.990834,0,1,0,0,1,0
4,0,3,0,35.0,1,1,0,2.202765,0,1,0,1,0,0
